# Lasso Regression - From Scratch (NumPy)

## What is Lasso Regression?

Lasso (Least Absolute Shrinkage and Selection Operator) Regression is a linear regression with L1 regularization. It adds a penalty equal to the absolute value of the magnitude of coefficients. This forces some coefficients to become exactly zero, performing automatic feature selection.

---

## Key Benefits

| Benefit | Description |
|---------|-------------|
| L1 Regularization | Adds L1 penalty to model weights |
| Feature Selection | Shrinks some coefficients to exactly zero |
| Sparsity | Produces sparse models with fewer features |
| Interpretability | Simplifies model by removing irrelevant features |

---

## Lasso vs Ridge

| Aspect | Ridge (L2) | Lasso (L1) |
|--------|------------|------------|
| Penalty | Sum of squared coefficients | Sum of absolute coefficients |
| Feature Selection | No (keeps all features) | Yes (can zero out features) |
| Solution Type | Closed-form | Coordinate descent |
| Use Case | Multicollinearity | Feature selection, sparse models |

---

## Lasso Formula

$$\text{Loss} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} |w_j|$$

Where:
- $\lambda$ is the regularization parameter (alpha)
- $w_j$ are the model coefficients

---

## Coordinate Descent Algorithm

Since Lasso has no closed-form solution, we use coordinate descent:

For each coefficient $w_j$:

$$w_j = \frac{S(\sum_i x_{ij}(y_i - \hat{y}_i^{(j)}), \lambda)}{\sum_i x_{ij}^2}$$

Where $S$ is the soft-thresholding operator:

$$S(z, \lambda) = \text{sign}(z) \cdot \max(|z| - \lambda, 0)$$

---

## Applications

| Domain | Use Case |
|--------|----------|
| Feature Selection | Identifying important features |
| High-Dimensional Data | When features exceed samples |
| Genomics | Gene selection in microarray data |
| Signal Processing | Sparse signal reconstruction |

---

## Advantages

- Automatic Feature Selection: Sets some coefficients to zero
- Sparse Models: Simpler and more interpretable
- Handles High Dimensions: Works when p > n
- Reduces Overfitting: Shrinks coefficients

---

## Limitations

- No Closed-Form Solution: Requires iterative methods
- Correlation Issues: Tends to select one of correlated features
- Hyperparameter Sensitivity: Requires careful alpha tuning
- Instability: Can be unstable with highly correlated features

---

## One-Line Summary

**Lasso Regression adds L1 penalty to linear regression, performing feature selection by shrinking some coefficients to zero.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Lasso

print("="*50)
print("LASSO REGRESSION - FROM SCRATCH")
print("="*50)

## Lasso Regression Class from Scratch

In [ ]:
class LassoRegression:
    """
    Lasso Regression implementation from scratch using coordinate descent
    """
    
    def __init__(self, alpha=1.0, max_iter=1000, tol=1e-4):
        self.alpha = alpha  # Regularization parameter (lambda)
        self.max_iter = max_iter
        self.tol = tol
        self.coef_ = None
        self.intercept_ = None
    
    def _soft_threshold(self, z, lambda_):
        """Soft thresholding operator for Lasso"""
        return np.sign(z) * np.maximum(np.abs(z) - lambda_, 0)
    
    def fit(self, X, y):
        """
        Fit Lasso Regression using coordinate descent
        """
        X = np.array(X)
        y = np.array(y)
        
        n_samples, n_features = X.shape
        
        # Initialize coefficients
        self.coef_ = np.zeros(n_features)
        self.intercept_ = np.mean(y)
        
        # Center y
        y_centered = y - self.intercept_
        
        # Coordinate descent
        for iteration in range(self.max_iter):
            coef_old = self.coef_.copy()
            
            for j in range(n_features):
                # Compute residual without feature j
                residual = y_centered - X @ self.coef_
                
                # Add back contribution of feature j
                residual += self.coef_[j] * X[:, j]
                
                # Compute gradient
                rho = X[:, j] @ residual
                
                # Update coefficient using soft thresholding
                self.coef_[j] = self._soft_threshold(rho, self.alpha) / (X[:, j] @ X[:, j])
            
            # Check convergence
            if np.max(np.abs(self.coef_ - coef_old)) < self.tol:
                break
        
        return self
    
    def predict(self, X):
        """Make predictions"""
        X = np.array(X)
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        """Calculate R2 score"""
        y_pred = self.predict(X)
        return r2_score(y, y_pred)

In [ ]:
# Create synthetic dataset
np.random.seed(0)
n_samples = 200
n_features = 10

X = np.random.randn(n_samples, n_features)

# Only first 5 features are important (sparse coefficients)
true_coef = np.array([3.2, -1.5, 0.7, 0, 2.8, 0, 0, 0, 0, 0])
y = X.dot(true_coef) + np.random.randn(n_samples) * 0.6

print("Dataset created:")
print(f"Samples: {n_samples}")
print(f"Features: {n_features}")
print(f"True coefficients (only first 5 non-zero): {true_coef}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
# Train Lasso from scratch
lasso_scratch = LassoRegression(alpha=0.1)
lasso_scratch.fit(X_train, y_train)

y_pred_scratch = lasso_scratch.predict(X_test)
mse_scratch = mean_squared_error(y_test, y_pred_scratch)
r2_scratch = r2_score(y_test, y_pred_scratch)

print("Lasso Regression (From Scratch) Results:")
print(f"MSE: {mse_scratch:.4f}")
print(f"R2 Score: {r2_scratch:.4f}")
print(f"Coefficients: {lasso_scratch.coef_}")
print(f"Intercept: {lasso_scratch.intercept_:.4f}")

In [ ]:
# Compare with sklearn Lasso
lasso_sklearn = Lasso(alpha=0.1, max_iter=10000)
lasso_sklearn.fit(X_train, y_train)
y_pred_sklearn = lasso_sklearn.predict(X_test)
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
r2_sklearn = r2_score(y_test, y_pred_sklearn)

print("\n" + "="*50)
print("Comparison: From Scratch vs Scikit-Learn")
print("="*50)
print(f"{'Method':<15} {'MSE':<12} {'R2 Score':<12}")
print("-"*50)
print(f"{'From Scratch':<15} {mse_scratch:<12.4f} {r2_scratch:<12.4f}")
print(f"{'Scikit-Learn':<15} {mse_sklearn:<12.4f} {r2_sklearn:<12.4f}")

In [ ]:
# Feature selection demonstration
print("\n" + "="*50)
print("Feature Selection Analysis")
print("="*50)

print(f"{'Feature':<10} {'True':<10} {'Lasso Coef':<12} {'Selected':<10}")
print("-"*50)
for i in range(n_features):
    selected = "Yes" if np.abs(lasso_scratch.coef_[i]) > 0.001 else "No"
    print(f"{'X'+str(i+1):<10} {true_coef[i]:<10.2f} {lasso_scratch.coef_[i]:<12.4f} {selected:<10}")

In [ ]:
# Effect of different alpha values on feature selection
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5]
coef_path = []

print("\n" + "="*50)
print("Effect of Alpha on Feature Selection")
print("="*50)

for alpha in alphas:
    lasso = LassoRegression(alpha=alpha)
    lasso.fit(X_train, y_train)
    coef_path.append(lasso.coef_)
    non_zero = np.sum(np.abs(lasso.coef_) > 0.001)
    print(f"Alpha = {alpha:5}: Non-zero coefficients = {non_zero}/{n_features}")

In [ ]:
# Visualize coefficient path
coef_path = np.array(coef_path)
plt.figure(figsize=(10, 6))

for i in range(n_features):
    plt.plot(alphas, coef_path[:, i], marker='o', label=f'Feature {i+1}')

plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path vs Alpha')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Lasso vs Ridge comparison on feature selection
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)

print("\n" + "="*50)
print("Lasso vs Ridge: Feature Selection Comparison")
print("="*50)

print(f"{'Feature':<10} {'True':<10} {'Lasso':<12} {'Ridge':<12}")
print("-"*50)
for i in range(n_features):
    print(f"{'X'+str(i+1):<10} {true_coef[i]:<10.2f} {lasso_scratch.coef_[i]:<12.4f} {ridge.coef_[i]:<12.4f}")

In [ ]:
# Day Completed
print("\n" + "="*50)
print("LASSO REGRESSION - FROM SCRATCH COMPLETED")
print("="*50)
print("Topics covered:")
print("- Lasso Regression definition and benefits")
print("- L1 regularization vs L2 regularization")
print("- Coordinate descent algorithm")
print("- Soft thresholding operator")
print("- Feature selection capability")
print("- Lasso vs Ridge comparison")
print("- Coefficient path visualization")
print("="*50)"
print("\nNext: Lasso Regression with Scikit-Learn")